# Probability Distributions and Estimates

## Complete beginner-to-advanced revision notes

This pack is written for a learner who knows basic arithmetic and has seen mean and standard deviation, but is new to probability distributions.

### How to study this notebook

1. Read the **core idea** first.
2. Learn the **conditions** before memorising a formula.
3. Work through the example by hand.
4. Run the short code cell.
5. Use the **common mistakes** section to catch exam traps.
6. Finish with the master cheatsheet and revision questions.

### Symbol key

| Symbol | Meaning |
|---|---|
| $X$ | A random variable: a number produced by a random process |
| $x$ or $k$ | One possible value of $X$ |
| $P(\cdot)$ | Probability of an event |
| $E[X]$ | Expected value or long-run mean |
| $\mu$ | Population mean |
| $\sigma$ | Population standard deviation |
| $\sigma^2$ | Population variance |
| $n$ | Number of trials or sample size; check the context |
| $p$ | Probability of success |
| $q$ | Probability of failure, $q=1-p$ |
| $\lambda$ | Expected Poisson count in the stated interval |

> **Big picture:** A probability distribution is a rule that tells us which values a random variable can take and how likely different values or ranges are.


## Source map and accuracy note

These notes teach all ideas, formulas, diagrams, and worked examples present in the supplied material. They also add the missing links needed to move from beginner understanding to confident use.

| Chapter | Supplied source used |
|---|---|
| 01-02 | `2-finalpdf,pmf,cdf (1).pdf`, pages 1-3 |
| 03 | `3-Bernoulli Distribution.pdf`, pages 1-3 |
| 04 | `4-Binomial Distribution.pdf`, pages 1-3; transcript lines 1-554 |
| 05 | `5-Poisson Distribution.pdf`, pages 1-2; transcript lines 555-1058 |
| 06 | `6-Normal Guassian Distribution.pdf`, pages 1-2; transcript lines 1059-1515 |
| 07 | `7-Standard Normal Distribution And Z score.pdf`, pages 1-2; transcript lines 1516-2064 |
| 08 | `8-Uniform Distribution.pdf`, pages 1-3; transcript lines 2065-2557 |
| 09 | `9-Log Normal Distribution.pdf`, pages 1-2; transcript lines 2558-2904 |
| 10 | `10-Power Law Distribution.pdf`, pages 1-2; transcript lines 2905-3175 |
| 11 | `11-Pareto Distribution.pdf`, pages 1-2; transcript lines 3176-3397 |
| 12 | `12-Central Limit Theorem.pdf`, pages 1-2; transcript lines 3398-3776 |
| 13 | `13-Estimates.pdf`, page 1; transcript lines 3777-3990 |

> **Accuracy policy:** the lecture's intended idea is preserved, but small mathematical slips are corrected and labelled. This matters because probabilities above 1 are not probabilities; they are warning sirens.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "font.size": 11})
rng = np.random.default_rng(42)


# 01 - The Relationship Between PMF, PDF and CDF

**Source alignment:** `2-finalpdf,pmf,cdf (1).pdf`, pages 1-2. The source shows a fair-die PMF and step-shaped CDF, then a continuous density and smooth CDF whose slope gives the density.

## Learning goals

- Tell a discrete random variable from a continuous one.
- Know when to use a PMF, PDF, or CDF.
- Move from a PMF/PDF to a CDF and back.
- Calculate point and interval probabilities correctly.


## 1. Start with a random variable

A **random variable** converts an uncertain outcome into a number.

- Die roll: $X\in\{1,2,3,4,5,6\}$ - discrete.
- Waiting time: $X\in[0,\infty)$ - continuous.
- A discrete variable has separate, countable values.
- A continuous variable can take any real value in an interval.

The **support** is the set of values that are possible.

## 2. The three functions

| Function | Used for | Meaning | Probability rule |
|---|---|---|---|
| PMF $p_X(x)$ | Discrete $X$ | Probability at the exact value $x$ | $P(X=x)=p_X(x)$ |
| PDF $f_X(x)$ | Continuous $X$ | Density near $x$ | $P(a\le X\le b)=\int_a^b f_X(x)\,dx$ |
| CDF $F_X(x)$ | Any $X$ | Probability accumulated up to $x$ | $F_X(x)=P(X\le x)$ |

> PMF values are probabilities. PDF heights are densities. A density may be greater than 1; only its **area** must obey probability rules.


## 3. PMF rules

For a discrete variable:

$$p_X(x)\ge 0,\qquad \sum_x p_X(x)=1$$

For a fair die:

$$p_X(k)=\frac16,\quad k=1,2,3,4,5,6$$

Therefore:

$$P(X\le2)=P(X=1)+P(X=2)=\frac16+\frac16=\frac13$$

## 4. PDF rules

For a continuous variable:

$$f_X(x)\ge 0,\qquad \int_{-\infty}^{\infty} f_X(x)\,dx=1$$

The probability at one exact point is zero:

$$P(X=x)=0$$

This does **not** mean the value is impossible. It means a single point has zero width and therefore zero area.


## 5. CDF rules - valid for every distribution

$$F_X(x)=P(X\le x)$$

Every CDF:

- stays between 0 and 1;
- never decreases;
- approaches 0 as $x\to-\infty$;
- approaches 1 as $x\to\infty$;
- is right-continuous.

Interval probabilities come from subtraction:

$$P(a<X\le b)=F_X(b)-F_X(a)$$

Endpoint details matter for discrete variables. For continuous variables, $<$ and $\le$ give the same probability because point probabilities are zero.

## 6. How the functions connect

**Discrete:**

$$F_X(x)=\sum_{k\le x}p_X(k)$$

If the support contains consecutive integers:

$$p_X(k)=F_X(k)-F_X(k-1)$$

**Continuous:**

$$F_X(x)=\int_{-\infty}^{x}f_X(t)\,dt$$

Wherever the derivative exists:

$$f_X(x)=\frac{d}{dx}F_X(x)$$

This matches the source diagram: the CDF's **slope** is the PDF's height.


In [ ]:
# Discrete example: fair-die PMF and CDF
x = np.arange(1, 7)
pmf = np.repeat(1/6, 6)
cdf = np.cumsum(pmf)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].stem(x, pmf, basefmt=" ")
ax[0].set(title="Fair die: PMF", xlabel="x", ylabel="P(X=x)", ylim=(0, 1))
ax[1].step(np.r_[0, x, 7], np.r_[0, 0, cdf], where="post")
ax[1].set(title="Fair die: CDF", xlabel="x", ylabel="P(X<=x)", ylim=(-.03, 1.05))
plt.tight_layout()

print("P(X <= 2) =", pmf[:2].sum())
print("CDF at 2   =", cdf[1])


In [ ]:
# Continuous example: standard-normal PDF and CDF
x = np.linspace(-4, 4, 500)
pdf = stats.norm.pdf(x)
cdf = stats.norm.cdf(x)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(x, pdf)
ax[0].fill_between(x, 0, pdf, where=(x >= -1) & (x <= 1), alpha=.3)
ax[0].set(title="PDF: probability is area", xlabel="x", ylabel="density")
ax[1].plot(x, cdf)
ax[1].set(title="CDF: accumulated probability", xlabel="x", ylabel="F(x)", ylim=(-.03, 1.03))
plt.tight_layout()

p_between = stats.norm.cdf(1) - stats.norm.cdf(-1)
print(f"P(-1 <= X <= 1) = {p_between:.4f}")


## Advanced connection: mixed distributions

Some variables have both point masses and continuous regions. Example: an insurance payment may be exactly $0$ with positive probability, but positive payments vary continuously.

- The CDF still works without modification.
- A jump in the CDF equals a point probability.
- A smooth region's slope gives density.

## Common mistakes

- Saying “PDF = probability at $x$.” It is density, not point probability.
- Calling CDF “cumulative density function.” The correct term is **cumulative distribution function**.
- Adding PDF heights instead of calculating area.
- Forgetting that a discrete CDF is step-shaped.

## Quick recall

- **PMF:** exact probability for discrete values.
- **PDF:** area gives probability for continuous values.
- **CDF:** probability up to a value; works for both.
- **CDF from PMF/PDF:** cumulative sum or integral.
- **PMF/PDF from CDF:** jump difference or derivative.


# 02 - Types of Probability Distribution

**Source alignment:** `2-finalpdf,pmf,cdf (1).pdf`, page 3. The source lists Bernoulli, Binomial, Normal/Gaussian, Poisson, Log-normal, and Uniform, then classifies example house-price features as discrete or continuous.

## The first and most important split

| Type | Possible values | Described by | Examples |
|---|---|---|---|
| Discrete | Countable values | PMF and CDF | Rooms, defects, goals, coin result |
| Continuous | Any value in an interval | PDF and CDF | Height, weight, time, price |
| Mixed | Discrete masses plus continuous parts | CDF plus both components | Zero-inflated payment |

**A useful test:** if counting makes sense, it is often discrete. If measurement to finer precision makes sense, it is often continuous.


## Distribution family map

| Distribution | Type | Main question it answers | Parameters |
|---|---|---|---|
| Bernoulli | Discrete | Did one trial succeed? | $p$ |
| Binomial | Discrete | How many successes in fixed $n$ trials? | $n,p$ |
| Poisson | Discrete | How many events in a fixed exposure? | $\lambda$ |
| Discrete Uniform | Discrete | Which equally likely listed value occurs? | $a,b$ |
| Normal | Continuous | Is variation roughly symmetric around a centre? | $\mu,\sigma^2$ |
| Continuous Uniform | Continuous | Is every equal-width interval equally likely? | $a,b$ |
| Log-normal | Continuous, positive | Is the logarithm normal? | log-scale $\mu,\sigma^2$ |
| Pareto | Continuous, positive | Is there a power-law heavy tail above a minimum? | $x_m,\alpha$ |

**Power law** is broader than one named distribution. It describes a scaling relationship such as $y=Cx^{-\alpha}$. The Pareto distribution is a probability model with a power-law tail.


## Other useful ways to classify distributions

### 1. Parametric vs non-parametric

- **Parametric:** described by a fixed set of parameters, such as $N(\mu,\sigma^2)$.
- **Non-parametric:** shape is not forced into a small formula family, such as an empirical distribution or kernel density estimate.

### 2. Univariate vs multivariate

- **Univariate:** one random variable, such as height.
- **Multivariate:** several variables jointly, such as height and weight together.

### 3. Symmetry and skew

- **Symmetric:** left and right sides mirror each other; Normal is the classic example.
- **Right-skewed:** long right tail; Log-normal and Pareto are examples.
- **Left-skewed:** long left tail.

### 4. Tail weight

- **Light-tailed:** extreme values become rare quickly, such as Normal.
- **Heavy-tailed:** extreme values remain more plausible, such as Pareto.


## Choosing a distribution: a practical decision guide

1. **What does $X$ represent?** A binary result, count, measurement, or waiting time?
2. **What values are possible?** This is the support.
3. **How was the data generated?** Fixed trials, fixed interval, equal chances, additive noise, or multiplicative growth?
4. **What assumptions are reasonable?** Independence? Constant rate? Same probability?
5. **Does the shape agree?** Use plots as evidence, not as proof.
6. **Does the model predict sensible probabilities?** Check with held-out data or goodness-of-fit methods.

| Situation | First model to consider |
|---|---|
| One yes/no event | Bernoulli |
| Success count from fixed identical trials | Binomial |
| Event count in time/space/exposure | Poisson |
| Symmetric measurement around a centre | Normal |
| Bounded and equally dense | Continuous Uniform |
| Finite equally likely outcomes | Discrete Uniform |
| Positive right-skewed multiplicative quantity | Log-normal |
| Very heavy upper tail with scaling | Pareto/power law |

> A histogram's shape alone is not enough. The data-generating process is the real boss fight.


In [ ]:
# A visual family overview
fig, ax = plt.subplots(2, 3, figsize=(13, 7))

k = np.arange(0, 11)
ax[0, 0].stem([0, 1], stats.bernoulli.pmf([0, 1], .6), basefmt=" ")
ax[0, 0].set_title("Bernoulli(p=.6)")
ax[0, 1].stem(k, stats.binom.pmf(k, 10, .4), basefmt=" ")
ax[0, 1].set_title("Binomial(n=10,p=.4)")
ax[0, 2].stem(k, stats.poisson.pmf(k, 3), basefmt=" ")
ax[0, 2].set_title("Poisson(lambda=3)")

x = np.linspace(-4, 8, 500)
ax[1, 0].plot(x, stats.norm.pdf(x, 1, 1.2))
ax[1, 0].set_title("Normal")
ax[1, 1].plot(x, stats.uniform.pdf(x, 0, 5))
ax[1, 1].set_title("Continuous Uniform(0,5)")
positive = np.linspace(.01, 8, 500)
ax[1, 2].plot(positive, stats.lognorm.pdf(positive, s=.7, scale=np.exp(0)))
ax[1, 2].set_title("Log-normal")

for a in ax.flat:
    a.set_xlabel("possible value")
plt.tight_layout()


## Dataset example from the source

For a house-price dataset:

- house size: continuous;
- number of rooms: discrete;
- location: categorical, not a numerical probability distribution by itself;
- floor number: discrete;
- sea-side flag $\{0,1\}$: Bernoulli-style binary;
- price: usually treated as continuous.

Exploratory data analysis (EDA) and feature engineering begin by identifying these types correctly.

## Common mistakes

- Calling categories such as suburb names “continuous.”
- Picking a distribution only because a histogram looks similar.
- Forgetting that the same variable may need a different model in a different process.
- Confusing a parameter, such as $p$, with an observed statistic, such as $\hat p$.


# 03 - Bernoulli Distribution

**Source alignment:** `3-Bernoulli Distribution.pdf`, pages 1-3. The source covers binary outcomes, $p$ and $q$, the PMF, mean, median, mode, and variance.

## Core idea

A Bernoulli random variable records the result of **one** trial with two outcomes:

- success: $X=1$ with probability $p$;
- failure: $X=0$ with probability $q=1-p$.

Examples: head/tail, pass/fail, click/no click, defective/not defective.

$$X\sim\operatorname{Bernoulli}(p),\qquad 0\le p\le1$$


## PMF and CDF

A compact PMF formula is:

$$P(X=x)=p^x(1-p)^{1-x},\qquad x\in\{0,1\}$$

Check it:

- If $x=1$: $p^1(1-p)^0=p$.
- If $x=0$: $p^0(1-p)^1=1-p=q$.

The CDF is:

$$F(x)=\begin{cases}
0,&x<0\\
1-p,&0\le x<1\\
1,&x\ge1
\end{cases}$$

## Centre and spread

$$E[X]=p$$

$$\operatorname{Var}(X)=p(1-p)=pq$$

$$\operatorname{SD}(X)=\sqrt{p(1-p)}$$

The mean equals the long-run fraction of successes because the only values are 0 and 1.


## Median and mode - with edge cases

**Median:**

- $0$ is a median when $p<0.5$.
- $1$ is a median when $p>0.5$.
- When $p=0.5$, every value in $[0,1]$ satisfies a common mathematical median definition. Some software reports $0.5$; some restrict the answer to observed values.

**Mode:**

- mode $=0$ if $p<0.5$;
- mode $=1$ if $p>0.5$;
- both 0 and 1 are modes if $p=0.5$.

## Worked example from the source

A customer uses a new phone with probability $p=0.60$.

$$P(X=1)=0.60,\qquad P(X=0)=0.40$$

$$E[X]=0.60,\quad \operatorname{Var}(X)=0.60(0.40)=0.24$$

$$\operatorname{SD}(X)=\sqrt{0.24}\approx0.4899$$


In [ ]:
# Bernoulli PMF, simulation, and theoretical moments
p = 0.60
x = np.array([0, 1])
pmf = stats.bernoulli.pmf(x, p)

sample = rng.binomial(n=1, p=p, size=10_000)
print("PMF [P(0), P(1)] =", pmf)
print("Theoretical mean    =", p)
print("Simulated mean      =", sample.mean())
print("Theoretical variance=", p * (1 - p))
print("Simulated variance  =", sample.var())

plt.bar(x, pmf, width=.35)
plt.xticks([0, 1], ["failure (0)", "success (1)"])
plt.ylabel("probability")
plt.title("Bernoulli PMF, p=0.60")
plt.show()


## From beginner to advanced

### A sequence of Bernoulli trials

If $X_1,\ldots,X_n$ are independent Bernoulli variables with the same $p$, then:

$$S=X_1+\cdots+X_n\sim\operatorname{Binomial}(n,p)$$

### Likelihood and maximum likelihood estimate

For observed results $x_1,\ldots,x_n$:

$$L(p)=\prod_{i=1}^n p^{x_i}(1-p)^{1-x_i}$$

The value of $p$ that maximises this likelihood is:

$$\hat p=\bar x=\frac{\text{number of successes}}{n}$$

### Entropy

Uncertainty is highest when $p=0.5$. If $p$ is near 0 or 1, the result is easier to predict.

$$H(X)=-p\log p-(1-p)\log(1-p)$$

## Common mistakes

- Using Bernoulli for several trials; use Binomial for the success count.
- Treating “success” as “good.” Success simply means the outcome coded as 1.
- Forgetting $q=1-p$.
- Writing the support as every integer instead of only $\{0,1\}$.


# 04 - Binomial Distribution

**Source alignment:** `4-Binomial Distribution.pdf`, pages 1-3, and transcript lines 1-554. The supplied examples are five coin flips and defect inspection.

## Core idea

A Binomial variable counts the number of successes in a fixed number of Bernoulli trials.

$$X\sim\operatorname{Binomial}(n,p)$$

- $n$: fixed number of trials;
- $p$: success probability on every trial;
- $q=1-p$: failure probability;
- $X=k$: exactly $k$ successes, where $k=0,1,\ldots,n$.

A Bernoulli distribution is the special case $n=1$.


## When is Binomial valid? Use BINS

- **B - Binary:** each trial has two outcomes.
- **I - Independent:** one trial does not change another.
- **N - Number fixed:** $n$ is decided in advance.
- **S - Same probability:** the same $p$ applies to each trial.

If sampling without replacement from a small population, independence and constant $p$ may fail. A hypergeometric model may be better.

## PMF

$$P(X=k)=\binom nk p^k(1-p)^{n-k}$$

$$\binom nk=\frac{n!}{k!(n-k)!}$$

Meaning:

- $p^k$: probability of the $k$ successes;
- $(1-p)^{n-k}$: probability of the failures;
- $\binom nk$: number of arrangements containing $k$ successes.


## Mean, variance, standard deviation, and shape

$$E[X]=np$$

$$\operatorname{Var}(X)=np(1-p)=npq$$

$$\operatorname{SD}(X)=\sqrt{npq}$$

Shape:

- $p=0.5$: symmetric;
- $p<0.5$: usually right-skewed;
- $p>0.5$: usually left-skewed;
- larger $n$: smoother-looking PMF.

A mode is $\lfloor(n+1)p\rfloor$. If $(n+1)p$ is an integer, there are two adjacent modes: $(n+1)p-1$ and $(n+1)p$.


## Worked example 1 - exactly three heads

Five fair coin flips: $n=5$, $p=0.5$. Find $P(X=3)$.

$$P(X=3)=\binom53(0.5)^3(0.5)^2$$

$$=10(0.5)^5=0.3125$$

## Worked example 2 - quality control

Ten items are inspected. Each has a 10% defect probability. Find exactly two defects.

$$P(X=2)=\binom{10}{2}(0.1)^2(0.9)^8$$

$$\approx0.1937=19.37\%$$

> **Correction to the spoken transcript:** the valid answer is about **0.1937**, not 1.937. A probability cannot exceed 1.


## Exact, cumulative, and tail questions

- Exactly $k$: $P(X=k)$ - use the PMF.
- At most $k$: $P(X\le k)$ - use the CDF.
- Fewer than $k$: $P(X<k)=P(X\le k-1)$.
- At least $k$: $P(X\ge k)=1-P(X\le k-1)$.
- More than $k$: $P(X>k)=1-P(X\le k)$.
- Between $a$ and $b$: add PMF values or subtract CDF values.


In [ ]:
# Exact and cumulative Binomial probabilities
n, p = 10, 0.10
exact_two = stats.binom.pmf(2, n, p)
at_most_two = stats.binom.cdf(2, n, p)
at_least_two = stats.binom.sf(1, n, p)  # sf(1) = P(X > 1)

print(f"P(X = 2)  = {exact_two:.6f}")
print(f"P(X <= 2) = {at_most_two:.6f}")
print(f"P(X >= 2) = {at_least_two:.6f}")

k = np.arange(0, n + 1)
plt.stem(k, stats.binom.pmf(k, n, p), basefmt=" ")
plt.xlabel("number of defective items")
plt.ylabel("probability")
plt.title("Binomial(n=10, p=0.10)")
plt.show()


## Advanced connections and approximations

### Sum of Bernoulli variables

$$X=X_1+\cdots+X_n$$

with independent $X_i\sim\operatorname{Bernoulli}(p)$ gives $X\sim\operatorname{Binomial}(n,p)$.

### Estimating $p$

If $k$ successes are observed in $n$ trials, the maximum likelihood estimate is:

$$\hat p=\frac{k}{n}$$

### Poisson approximation

When $n$ is large, $p$ is small, and $\lambda=np$ is moderate:

$$\operatorname{Binomial}(n,p)\approx\operatorname{Poisson}(\lambda=np)$$

### Normal approximation

When both $np$ and $n(1-p)$ are sufficiently large, a Normal approximation can work:

$$X\approx N(np,npq)$$

Use a **continuity correction**. For example, approximate $P(X\le k)$ with $P(Y\le k+0.5)$.

## Common mistakes

- Using changing $p$ values but still calling the count Binomial.
- Forgetting the combination term $\binom nk$.
- Using $1-P(X\le k)$ for “at least $k$”; the correct complement is $1-P(X\le k-1)$.
- Confusing the count $X$ with the success probability $p$.


# 05 - Poisson Distribution

**Source alignment:** `5-Poisson Distribution.pdf`, pages 1-2, and transcript lines 555-1058. The source uses counts of people arriving at a bank or hospital during fixed time intervals.

## Core idea

A Poisson variable counts events in a fixed amount of exposure:

- time: calls per hour;
- space: defects per square metre;
- distance: accidents per kilometre;
- volume: particles per millilitre.

$$X\sim\operatorname{Poisson}(\lambda),\qquad \lambda>0$$

$\lambda$ is the **expected count in the stated interval**, not the minimum count.


## Conditions behind the basic Poisson model

- Events occur independently.
- The average event rate is constant over the interval.
- The chance of two or more events in a tiny interval is negligible.
- Counts in non-overlapping intervals are independent.

Real arrivals may violate these assumptions because of rush hours, seasonality, queues, or events arriving in clusters.

## PMF and CDF

$$P(X=k)=\frac{e^{-\lambda}\lambda^k}{k!},\qquad k=0,1,2,\ldots$$

$$F(k)=P(X\le k)=\sum_{j=0}^{k}\frac{e^{-\lambda}\lambda^j}{j!}$$

If the rate is $r$ events per unit and the exposure is $t$ units:

$$\lambda=rt$$

Example: 3 customers per hour for 2 hours gives $\lambda=6$ for the two-hour count.


## Mean, variance, and standard deviation

$$E[X]=\lambda$$

$$\operatorname{Var}(X)=\lambda$$

$$\operatorname{SD}(X)=\sqrt{\lambda}$$

The equality of mean and variance is a key diagnostic. If observed variance is much larger than the mean, the data are **overdispersed**; a negative-binomial model may fit better.

## Worked example from the source

Suppose the expected count is $\lambda=3$. Find exactly five events.

$$P(X=5)=\frac{e^{-3}3^5}{5!}\approx0.1008$$

That is about 10.08%.

For “four or five events”:

$$P(X=4\text{ or }5)=P(X=4)+P(X=5)$$

For “at most three events”:

$$P(X\le3)=\sum_{k=0}^3P(X=k)$$


In [ ]:
lam = 3
print(f"P(X = 5)  = {stats.poisson.pmf(5, lam):.6f}")
print(f"P(X <= 3) = {stats.poisson.cdf(3, lam):.6f}")
print(f"P(X >= 4) = {stats.poisson.sf(3, lam):.6f}")

k = np.arange(0, 18)
for current_lam in [1, 4, 10]:
    plt.plot(k, stats.poisson.pmf(k, current_lam), "o-", label=f"lambda={current_lam}")
plt.xlabel("event count k")
plt.ylabel("P(X=k)")
plt.title("Poisson PMFs")
plt.legend()
plt.show()


## Advanced connections

### Additivity

If independent $X\sim\operatorname{Poisson}(\lambda_1)$ and $Y\sim\operatorname{Poisson}(\lambda_2)$, then:

$$X+Y\sim\operatorname{Poisson}(\lambda_1+\lambda_2)$$

### Relation to Binomial

A Poisson model approximates many rare independent opportunities:

$$\operatorname{Binomial}(n,p)\to\operatorname{Poisson}(np)$$

when $n$ is large and $p$ is small.

### Inter-arrival time

In a Poisson process with rate $r$, the waiting time between events follows an Exponential distribution with mean $1/r$.

### Rate modelling

Poisson regression models counts while accounting for predictors and exposure. The log of exposure is often included as an **offset**.

## Common mistakes

- Treating $\lambda$ as a minimum rather than an average.
- Forgetting to rescale $\lambda$ when the interval changes.
- Using Poisson when the rate changes strongly over time.
- Using the model for bounded counts when the upper limit matters.


# 06 - Normal (Gaussian) Distribution

**Source alignment:** `6-Normal Guassian Distribution.pdf`, pages 1-2, and transcript lines 1059-1515. The source covers the bell curve, symmetry, parameters, PDF, Iris-style measurements, and the 68-95-99.7 rule.

## Core idea

The Normal distribution models a continuous variable whose values cluster symmetrically around a centre.

$$X\sim N(\mu,\sigma^2)$$

- $\mu\in\mathbb R$: centre or mean;
- $\sigma^2>0$: variance;
- $\sigma>0$: standard deviation;
- support: every real number.

For an exact Normal distribution:

$$\text{mean}=\text{median}=\text{mode}=\mu$$


## Shape and parameters

- The curve is symmetric around $\mu$.
- Half the probability lies below $\mu$ and half above.
- Larger $\sigma$ means a wider, flatter curve.
- Smaller $\sigma$ means a narrower, taller curve.
- Total area under the curve is 1.

Changing $\mu$ shifts the curve. Changing $\sigma$ changes its spread.

## PDF

$$f(x)=\frac{1}{\sigma\sqrt{2\pi}}
\exp\left[-\frac12\left(\frac{x-\mu}{\sigma}\right)^2\right]$$

The PDF height is not $P(X=x)$. Interval probability is area under the curve.

The Normal CDF has no simple elementary formula, so tables or software are normally used.


## The empirical rule: 68-95-99.7

For Normal $X$:

$$P(\mu-\sigma\le X\le\mu+\sigma)\approx0.6827$$

$$P(\mu-2\sigma\le X\le\mu+2\sigma)\approx0.9545$$

$$P(\mu-3\sigma\le X\le\mu+3\sigma)\approx0.9973$$

Approximate tail facts:

- outside $\mu\pm2\sigma$: about 4.55% total;
- above $\mu+2\sigma$: about 2.275%;
- outside $\mu\pm3\sigma$: about 0.27% total.

These statements apply when the Normal model is reasonable.


## Examples and modelling caution

Heights, measurement errors, and some biological measurements may be approximately Normal within a reasonably homogeneous group.

The supplied notes mention student height/weight and Iris measurements such as petal length. These can look bell-shaped in some subsets, but do not assume normality automatically.

> “Many data points in the universe are Normal” is too strong. The Normal distribution is common because sums of many small effects are often approximately Normal, not because every dataset obeys it.

## Mean and variance: population vs sample

Population formulas:

$$\mu=\frac1N\sum_{i=1}^N x_i,\qquad
\sigma^2=\frac1N\sum_{i=1}^N(x_i-\mu)^2$$

For a sample used to estimate population variance, the usual unbiased estimator is:

$$s^2=\frac1{n-1}\sum_{i=1}^n(x_i-\bar x)^2$$


In [ ]:
# Effect of mean and standard deviation
x = np.linspace(-8, 10, 700)
for mu, sigma in [(0, .6), (0, 1.5), (3, 1.0)]:
    plt.plot(x, stats.norm.pdf(x, mu, sigma), label=f"mu={mu}, sigma={sigma}")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Normal PDF: location and spread")
plt.legend()
plt.show()

for width in [1, 2, 3]:
    p = stats.norm.cdf(width) - stats.norm.cdf(-width)
    print(f"Within {width} standard deviation(s): {100*p:.3f}%")


In [ ]:
# A simple Q-Q plot without statsmodels
sample = rng.normal(loc=10, scale=2, size=300)
stats.probplot(sample, dist="norm", plot=plt)
plt.title("Normal Q-Q plot: points near a line support normality")
plt.show()


## Checking whether Normal is reasonable

Use several pieces of evidence:

- histogram or density plot;
- Normal Q-Q plot;
- domain knowledge;
- residual plots when checking a model;
- formal tests such as Shapiro-Wilk, used carefully.

Large datasets can make formal tests reject tiny, unimportant deviations. Visual and practical importance still matter.

## Advanced facts

- Linear transformation: if $X\sim N(\mu,\sigma^2)$, then $aX+b\sim N(a\mu+b,a^2\sigma^2)$.
- Sums of independent Normal variables are Normal.
- For Normal data, the MLE of $\mu$ is $\bar x$.
- The MLE of $\sigma^2$ uses divisor $n$; the usual unbiased sample variance uses $n-1$.

## Common mistakes

- Treating PDF height as probability.
- Writing $N(\mu,\sigma)$ when the convention expects variance. Always state your convention.
- Assuming a symmetric histogram proves normality.
- Applying the 68-95-99.7 rule to strongly skewed data.


# 07 - Standard Normal Distribution and Z-score

**Source alignment:** `7-Standard Normal Distribution And Z score.pdf`, pages 1-2, and transcript lines 1516-2064. The source converts measurements to a common scale and connects standardisation to machine learning.

## Standard Normal distribution

$$Z\sim N(0,1)$$

It is a Normal distribution with mean 0 and standard deviation 1.

Any Normal value can be standardised using:

$$z=\frac{x-\mu}{\sigma}$$

Reverse the transformation with:

$$x=\mu+z\sigma$$


## How to interpret a z-score

A z-score tells how many standard deviations a value is from the mean.

- $z=0$: exactly at the mean.
- $z=1.4$: 1.4 standard deviations above the mean.
- $z=-1.4$: 1.4 standard deviations below the mean.
- The sign gives direction; the magnitude gives distance.

## Worked examples from the source

If $\mu=4$ and $\sigma=1$:

$$z_{4.25}=\frac{4.25-4}{1}=0.25$$

So 4.25 is 0.25 standard deviations above the mean.

$$z_{2.5}=\frac{2.5-4}{1}=-1.5$$

So 2.5 is 1.5 standard deviations below the mean.

> In the source's teaching set $\{1,2,3,4,5\}$, the actual population SD is $\sqrt2\approx1.414$. The lecture temporarily uses $\sigma\approx1$ only to simplify the demonstration.


## Probabilities with the Standard Normal CDF

Let $\Phi(z)=P(Z\le z)$.

- Below $x$: $P(X\le x)=\Phi(z)$.
- Above $x$: $P(X>x)=1-\Phi(z)$.
- Between $a$ and $b$: $P(a<X<b)=\Phi(z_b)-\Phi(z_a)$.

Example: IQ is modelled as $N(100,15^2)$. For 130:

$$z=\frac{130-100}{15}=2$$

$$P(X>130)=1-\Phi(2)\approx0.0228$$

So about 2.28% lie above 130 under this model.


In [ ]:
mu, sigma = 100, 15
x = 130
z = (x - mu) / sigma
p_above = stats.norm.sf(z)
print(f"z-score = {z:.2f}")
print(f"P(X > {x}) = {p_above:.4f}")

z_grid = np.linspace(-4, 4, 500)
plt.plot(z_grid, stats.norm.pdf(z_grid))
plt.fill_between(z_grid, 0, stats.norm.pdf(z_grid), where=(z_grid > z), alpha=.35)
plt.axvline(z, color="black", linestyle="--")
plt.title("Standard Normal upper-tail probability")
plt.xlabel("z")
plt.ylabel("density")
plt.show()


## Standardisation in data science

For a feature with observations $x_i$:

$$z_i=\frac{x_i-\bar x}{s}$$

This puts age, weight, height, and salary onto comparable unit-free scales.

It is often useful for:

- distance-based methods such as k-means and k-nearest neighbours;
- regularised linear/logistic regression;
- principal component analysis;
- neural-network optimisation.

Tree-based models usually do not need standardisation because split ordering is unchanged.

### Avoid data leakage

1. Compute mean and SD using the training set only.
2. Use those same training values to transform validation and test data.

Do not fit the scaler on all data before splitting.


In [ ]:
# Manual standardisation of differently scaled features
df = pd.DataFrame({
    "age": [24, 25, 26, 27, 30, 31],
    "weight_kg": [70, 60, 55, 40, 30, 25],
    "height_cm": [175, 160, 150, 130, 175, 180],
    "salary_k": [40, 50, 60, 30, 20, 70],
})

z_df = (df - df.mean()) / df.std(ddof=0)
print(z_df.round(2).to_string())
print("Means after standardisation:\n", z_df.mean().round(10))
print("Population SDs after standardisation:\n", z_df.std(ddof=0).round(10))


## Important distinctions

- A **population z-score** uses known $\mu$ and $\sigma$.
- A **sample standard score** often uses $\bar x$ and $s$.
- A **z test statistic** is a different object used in inference.
- Z-scoring changes units and centre/spread, but it does **not** make a non-Normal variable Normal.
- Extreme z-scores may flag outliers only when the model and context support that conclusion.

## Common mistakes

- Forgetting to subtract the mean first.
- Dropping the negative sign for values below the mean.
- Saying $z=2$ means the value is “two units” above; it means two **standard deviations** above.
- Assuming values must lie between -3 and 3. Normal values can lie beyond them; they are simply rare.


# 08 - Uniform Distribution

**Source alignment:** `8-Uniform Distribution.pdf`, pages 1-3, and transcript lines 2065-2557. The source covers both continuous and discrete Uniform distributions, their formulas, and a candy-sales example.

## Core idea

“Uniform” means equal chances in the correct sense:

- **Continuous Uniform:** equal-length subintervals have equal probability.
- **Discrete Uniform:** every listed outcome has equal probability.

These are related but not interchangeable.


## A. Continuous Uniform distribution

$$X\sim U(a,b),\qquad a<b$$

Support: $a\le X\le b$.

### PDF

$$f(x)=\begin{cases}
\frac1{b-a},&a\le x\le b\\
0,&\text{otherwise}
\end{cases}$$

### CDF

$$F(x)=\begin{cases}
0,&x<a\\
\frac{x-a}{b-a},&a\le x\le b\\
1,&x>b
\end{cases}$$

### Centre and spread

$$E[X]=\frac{a+b}{2},\qquad \operatorname{Median}(X)=\frac{a+b}{2}$$

$$\operatorname{Var}(X)=\frac{(b-a)^2}{12}$$


## Continuous worked example from the source

Model daily candy sales as $U(10,40)$.

### Between 15 and 30

Probability is the wanted width divided by total width:

$$P(15\le X\le30)=\frac{30-15}{40-10}=\frac{15}{30}=0.5$$

### Greater than 20

$$P(X>20)=\frac{40-20}{40-10}=\frac{20}{30}=0.6667$$

> **Modelling note:** a literal number of candies is an integer, so a discrete model is more exact. The source intentionally uses a continuous rectangle to teach area calculations.


In [ ]:
a, b = 10, 40
p_15_to_30 = stats.uniform.cdf(30, loc=a, scale=b-a) - stats.uniform.cdf(15, loc=a, scale=b-a)
p_above_20 = stats.uniform.sf(20, loc=a, scale=b-a)
print(f"P(15 <= X <= 30) = {p_15_to_30:.4f}")
print(f"P(X > 20)         = {p_above_20:.4f}")

x = np.linspace(5, 45, 500)
pdf = stats.uniform.pdf(x, loc=a, scale=b-a)
cdf = stats.uniform.cdf(x, loc=a, scale=b-a)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(x, pdf); ax[0].set_title("Continuous Uniform PDF")
ax[1].plot(x, cdf); ax[1].set_title("Continuous Uniform CDF")
for current in ax: current.set_xlabel("x")
plt.tight_layout()


## B. Discrete Uniform distribution

Suppose possible integer values are $a,a+1,\ldots,b$.

$$n=b-a+1$$

$$P(X=k)=\frac1n,\qquad k\in\{a,\ldots,b\}$$

For a fair die, $a=1$, $b=6$, $n=6$, so every face has probability $1/6$.

$$E[X]=\frac{a+b}{2}$$

$$\operatorname{Var}(X)=\frac{n^2-1}{12}$$

For integer $x$ within the support:

$$F(x)=\frac{\lfloor x\rfloor-a+1}{n}$$

with 0 below $a$ and 1 at or above $b$.


In [ ]:
# Fair die as a discrete Uniform distribution
outcomes = np.arange(1, 7)
probabilities = np.repeat(1/6, 6)
print("P(2 <= X <= 5) =", probabilities[1:5].sum())
print("Mean =", outcomes.mean(), "Variance =", outcomes.var())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].stem(outcomes, probabilities, basefmt=" ")
ax[0].set(title="Discrete Uniform PMF", xlabel="die face", ylabel="probability")
ax[1].step(outcomes, np.cumsum(probabilities), where="post")
ax[1].set(title="Discrete Uniform CDF", xlabel="die face", ylabel="cumulative probability")
plt.tight_layout()


## Advanced facts

- If $U\sim U(0,1)$ and $F$ is a continuous CDF, then $X=F^{-1}(U)$ has CDF $F$. This is inverse-transform sampling.
- A Uniform model is often used when only bounds are known and no location within the interval is preferred.
- A cryptographic random generator needs stronger guarantees than merely matching a Uniform histogram.

## Common mistakes

- Saying every exact real value in a continuous Uniform distribution has positive probability.
- Forgetting the $+1$ in the number of discrete integer outcomes, $n=b-a+1$.
- Using $b-a$ as a discrete count.
- Assuming a bounded variable must be Uniform.


# 09 - Log-normal Distribution

**Source alignment:** `9-Log Normal Distribution.pdf`, pages 1-2, and transcript lines 2558-2904. The source shows the right-skewed shape, log transformation, Q-Q checking, and examples including wealth, comment length, chess-game length, dwell time, and salaries.

## Core idea

A positive random variable $X$ is Log-normal when its natural logarithm is Normal:

$$X\sim\operatorname{LogNormal}(\mu,\sigma^2)
\quad\Longleftrightarrow\quad
Y=\ln X\sim N(\mu,\sigma^2)$$

Equivalently:

$$X=e^Y$$

Support: $x>0$. The distribution is usually right-skewed.

> $\mu$ and $\sigma$ describe **$\ln X$**, not the original-scale mean and standard deviation of $X$.


## PDF and CDF

$$f(x)=\frac{1}{x\sigma\sqrt{2\pi}}
\exp\left[-\frac{(\ln x-\mu)^2}{2\sigma^2}\right],\qquad x>0$$

$$F(x)=\Phi\left(\frac{\ln x-\mu}{\sigma}\right),\qquad x>0$$

where $\Phi$ is the Standard Normal CDF.

## Original-scale summaries

$$E[X]=e^{\mu+\sigma^2/2}$$

$$\operatorname{Median}(X)=e^\mu$$

$$\operatorname{Mode}(X)=e^{\mu-\sigma^2}$$

$$\operatorname{Var}(X)=\left(e^{\sigma^2}-1\right)e^{2\mu+\sigma^2}$$

For $\sigma>0$, mean $>$ median $>$ mode.


## Why Log-normal appears

Normal models often arise from **additive** effects. Log-normal models often arise from **multiplicative** effects.

If:

$$X=A_1A_2\cdots A_m$$

then:

$$\ln X=\ln A_1+\ln A_2+\cdots+\ln A_m$$

A sum of many small effects may be approximately Normal, making the product approximately Log-normal.

Plausible examples:

- task completion and dwell times;
- sizes produced by multiplicative growth;
- some incomes or salaries;
- comment or file sizes.

Wealth may have a Log-normal-like body and a Pareto-like extreme tail; no single family automatically fits the whole range.


In [ ]:
mu, sigma = 0.5, 0.8  # parameters of ln(X)
x = np.linspace(.01, 12, 700)
dist = stats.lognorm(s=sigma, scale=np.exp(mu))

print(f"Mean   = {dist.mean():.4f}")
print(f"Median = {dist.median():.4f}")
print(f"P(X>5) = {dist.sf(5):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(x, dist.pdf(x))
ax[0].set(title="Log-normal PDF", xlabel="x", ylabel="density")

sample = rng.lognormal(mean=mu, sigma=sigma, size=2_000)
ax[1].hist(np.log(sample), bins=35, density=True, alpha=.55)
y = np.linspace(np.log(sample).min(), np.log(sample).max(), 400)
ax[1].plot(y, stats.norm.pdf(y, mu, sigma))
ax[1].set(title="ln(X) is Normal", xlabel="ln(x)", ylabel="density")
plt.tight_layout()


## How to assess a Log-normal model

1. Check that observations are positive.
2. Compute $y_i=\ln x_i$.
3. Plot a histogram and Normal Q-Q plot of $y$.
4. Check domain assumptions and tail fit.
5. Compare alternatives using predictive performance or likelihood-based criteria.

A good Q-Q line for $\ln X$ supports, but does not prove, a Log-normal model.

## Estimation

If $X_i$ are Log-normal, estimate log-scale parameters from $Y_i=\ln X_i$:

$$\hat\mu=\frac1n\sum_i\ln X_i$$

$$\hat\sigma^2_{\text{MLE}}=\frac1n\sum_i(\ln X_i-\hat\mu)^2$$

## Common mistakes

- Allowing zero or negative values without a justified model change.
- Treating log-scale $\mu$ as the original mean.
- Believing a log transform always makes data Normal.
- Reporting only the arithmetic mean when strong skew makes median and quantiles more informative.


# 10 - Power-law Distribution

**Source alignment:** `10-Power Law Distribution.pdf`, pages 1-2, and transcript lines 2905-3175. The source introduces the long tail, the 80/20 idea, examples from sport, wealth, oil, word frequency, and software defects, plus transformations.

## Core idea

A power law is a scaling relationship:

$$y=Cx^{-\alpha}$$

where $C>0$ and usually $\alpha>0$.

If $x$ is multiplied by a factor $c$:

$$y(cx)=c^{-\alpha}y(x)$$

The proportional change depends on $c$ and $\alpha$, not on the starting size of $x$. This is called **scale invariance**.

> “Power law” may describe a relationship, a probability density, a frequency pattern, or a tail. Always state which one you mean.


## Shape and long-tail behaviour

For $\alpha>0$, $Cx^{-\alpha}$ falls quickly at first and then slowly.

- Small values are common.
- Very large values are rare but not as rare as under a Normal model.
- The long tail can make extremes matter greatly.

On log-log axes:

$$\log y=\log C-\alpha\log x$$

This is a straight line with slope $-\alpha$ for an exact power law.

A straight-looking log-log plot is only an initial clue; several heavy-tailed models can look similar over a limited range.


## The 80/20 idea - useful, but not the definition

The source connects power laws with the Pareto principle:

- a small fraction of players or teams may produce much of the result;
- a minority of people may hold most wealth;
- a small set of words may account for most usage;
- a small set of software defects may cause many failures.

“80/20” is a memorable pattern, not a universal mathematical law. A power law can produce 70/30, 90/10, or another concentration depending on its exponent and measurement.

## Common empirical examples

- word-frequency ranks (Zipf-like behaviour);
- city sizes over some ranges;
- network degrees;
- file sizes and event magnitudes;
- the upper tail of wealth or income.

Each claim needs evidence over a defensible range.


In [ ]:
# Exact power law: y = C*x^(-alpha)
C, alpha = 5, 2.2
x = np.logspace(0, 3, 300)
y = C * x ** (-alpha)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(x, y)
ax[0].set(title="Ordinary axes", xlabel="x", ylabel="y")
ax[1].loglog(x, y)
ax[1].set(title="Log-log axes: slope = -alpha", xlabel="x (log)", ylabel="y (log)")
plt.tight_layout()

estimated_slope = np.polyfit(np.log(x), np.log(y), 1)[0]
print(f"Estimated log-log slope = {estimated_slope:.3f}")


## Power-law probability tails

A common tail statement is:

$$P(X>x)\propto x^{-\beta}$$

This survival-function exponent is not always the same symbol or numerical exponent used for the PDF. For Pareto Type I:

$$P(X>x)=\left(\frac{x_m}{x}\right)^\alpha$$

while its PDF decays as $x^{-(\alpha+1)}$.

### Why moments may fail to exist

Extremely heavy tails can make theoretical summaries infinite:

- a Pareto mean is finite only for $\alpha>1$;
- its variance is finite only for $\alpha>2$.

Sample averages can therefore be unstable.


## How to investigate a possible power law

1. Identify a plausible lower threshold $x_{min}$.
2. Plot the empirical survival function on log-log axes.
3. Estimate the exponent, preferably by maximum likelihood.
4. Check goodness of fit, often with a KS-style distance and simulation.
5. Compare alternatives such as Log-normal, Exponential, and stretched Exponential.
6. Report the fitted range; do not claim a law for the entire dataset if only the tail fits.

### Transformation warning

A Box-Cox transformation may reduce skewness for positive data. It does **not** guarantee that every power-law dataset becomes Normal. Check the transformed distribution and confirm that transformation makes sense for the task.

## Common mistakes

- Defining power law as “the 80/20 rule.”
- Claiming a power law from a straight-looking plot alone.
- Mixing PDF and survival-function exponents.
- Ignoring the lower cutoff and finite-size limits.


# 11 - Pareto Distribution

**Source alignment:** `11-Pareto Distribution.pdf`, pages 1-2, and transcript lines 3176-3397. The source presents Pareto as a non-Gaussian power-law distribution, explains the role of $\alpha$, uses IT project/defect examples, and mentions Box-Cox transformation.

## Core idea

Pareto Type I models a positive quantity above a minimum value with a heavy upper tail.

$$X\sim\operatorname{Pareto}(x_m,\alpha)$$

- $x_m>0$: minimum possible value or scale;
- $\alpha>0$: shape or tail index;
- support: $x\ge x_m$.

Smaller $\alpha$ means a heavier tail and more extreme inequality. Larger $\alpha$ means faster tail decay.


## PDF, CDF, survival function, and quantile

$$f(x)=\frac{\alpha x_m^\alpha}{x^{\alpha+1}},\qquad x\ge x_m$$

$$F(x)=1-\left(\frac{x_m}{x}\right)^\alpha$$

The survival function gives the chance of exceeding $x$:

$$P(X>x)=\left(\frac{x_m}{x}\right)^\alpha$$

Quantile function for $0<p<1$:

$$Q(p)=x_m(1-p)^{-1/\alpha}$$

At the lower boundary:

$$f(x_m)=\frac{\alpha}{x_m}$$

so increasing $\alpha$ raises the starting PDF height while making the tail decay faster.


## Mean, variance, median, and mode

$$E[X]=\frac{\alpha x_m}{\alpha-1}\quad\text{only if }\alpha>1$$

$$\operatorname{Var}(X)=\frac{\alpha x_m^2}{(\alpha-1)^2(\alpha-2)}
\quad\text{only if }\alpha>2$$

$$\operatorname{Median}(X)=x_m2^{1/\alpha}$$

$$\operatorname{Mode}(X)=x_m$$

If $\alpha\le1$, the theoretical mean is infinite. If $\alpha\le2$, the theoretical variance is infinite.


## Worked example

Let $x_m=10$ and $\alpha=2$. Find the chance that $X>30$.

$$P(X>30)=\left(\frac{10}{30}\right)^2=\frac19\approx0.1111$$

Find the median:

$$Q(0.5)=10(0.5)^{-1/2}=10\sqrt2\approx14.14$$

The mean is $20$, but the variance is infinite because $\alpha=2$ is not greater than 2. This is a good example of why a finite sample can look calm while the theoretical tail is chaos wearing a tie.


In [ ]:
xm, alpha = 10, 2
# scipy.stats.pareto uses shape b=alpha and scale=xm
dist = stats.pareto(b=alpha, scale=xm)
print(f"P(X > 30) = {dist.sf(30):.4f}")
print(f"Median    = {dist.median():.4f}")
print(f"Mean      = {dist.mean():.4f}")
print("Variance  =", dist.var())

x = np.linspace(xm, 80, 700)
for a in [1.2, 2, 4]:
    plt.plot(x, stats.pareto.pdf(x, b=a, scale=xm), label=f"alpha={a}")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Pareto Type I PDF")
plt.legend()
plt.show()


## Pareto distribution and the 80/20 rule

The Pareto family does not always produce an exact 80/20 split.

For a Pareto distribution with $\alpha>1$, the share held by the largest fraction $u$ is:

$$S_{top}(u)=u^{1-1/\alpha}$$

Exact 80/20 concentration means $S_{top}(0.2)=0.8$, which gives approximately:

$$\alpha\approx1.161$$

Different $\alpha$ values produce different concentration levels.

## Maximum likelihood estimate of $\alpha$

If $x_m$ is fixed and all observations satisfy $x_i\ge x_m$:

$$\hat\alpha=\frac{n}{\sum_{i=1}^n\ln(x_i/x_m)}$$

Threshold selection is important; fitting the formula to the wrong range gives a misleading tail model.


## Pareto vs Log-normal

| Feature | Pareto | Log-normal |
|---|---|---|
| Support | $x\ge x_m$ | $x>0$ |
| Tail | Power law | Heavy-ish but decays faster than Pareto |
| Log-log survival | Straight for exact model | Curves eventually |
| Moments | May be infinite | All positive moments finite |

Both can look right-skewed, so compare them rather than deciding by eye.

## Transformation warning and common mistakes

- Box-Cox can reduce skewness, but it does not automatically create Normal data.
- Pareto is a specific probability distribution; power law is a broader scaling idea.
- The 80/20 principle is not guaranteed.
- A few extreme observations are not enough to prove a Pareto tail.


# 12 - Central Limit Theorem

**Source alignment:** `12-Central Limit Theorem.pdf`, pages 1-2, and transcript lines 3398-3776. The source draws repeated samples, plots their means, and gives the sampling mean and standard error.

## The problem CLT solves

Suppose a population has:

- mean $\mu$;
- finite variance $\sigma^2$.

Take many independent random samples of size $n$. Compute each sample mean $\bar X$.

The set of those means has a **sampling distribution**.

The Central Limit Theorem says that, under suitable conditions, this sampling distribution becomes approximately Normal as $n$ grows.


## Mathematical statement

For independent and identically distributed observations with finite mean and variance:

$$\frac{\bar X-\mu}{\sigma/\sqrt n}\xrightarrow{d}N(0,1)$$

Equivalently, for sufficiently large $n$:

$$\bar X\approx N\left(\mu,\frac{\sigma^2}{n}\right)$$

Therefore:

$$E[\bar X]=\mu$$

$$\operatorname{Var}(\bar X)=\frac{\sigma^2}{n}$$

$$\operatorname{SD}(\bar X)=\frac{\sigma}{\sqrt n}$$

The last quantity is the **standard error of the mean**.


## What becomes Normal?

The distribution of **sample means** becomes approximately Normal.

The CLT does **not** say:

- the original population becomes Normal;
- one sample becomes Normal;
- every statistic has the same result;
- $n=30$ guarantees accuracy.

If the population is exactly Normal, $\bar X$ is exactly Normal for every $n$.

If the population is skewed, larger $n$ is needed. For extremely heavy tails or dependence, the usual CLT may fail or require stronger conditions.


## Why the means become less variable

Individual observations vary with SD $\sigma$. A sample mean averages $n$ observations, so random ups and downs partly cancel.

$$SE(\bar X)=\frac{\sigma}{\sqrt n}$$

Consequences:

- multiplying $n$ by 4 halves the standard error;
- multiplying $n$ by 100 makes the standard error one tenth as large;
- doubling $n$ does **not** halve the error.

This square-root law matters in experimental design and survey sample sizes.


In [ ]:
# CLT simulation using a strongly right-skewed Exponential population
sample_sizes = [1, 5, 30, 100]
repetitions = 20_000
fig, ax = plt.subplots(2, 2, figsize=(11, 7))

for current_ax, n in zip(ax.flat, sample_sizes):
    means = rng.exponential(scale=1, size=(repetitions, n)).mean(axis=1)
    current_ax.hist(means, bins=60, density=True, alpha=.65)
    grid = np.linspace(means.min(), means.max(), 400)
    current_ax.plot(grid, stats.norm.pdf(grid, loc=1, scale=1/np.sqrt(n)), color="black")
    current_ax.set_title(f"Sampling means, n={n}")
    current_ax.set_xlabel("sample mean")

plt.tight_layout()


In [ ]:
# Verify the mean and standard error numerically
population_mu, population_sigma, n = 1, 1, 64
means = rng.exponential(scale=population_mu, size=(50_000, n)).mean(axis=1)

print(f"Mean of sample means: {means.mean():.4f} (theory {population_mu})")
print(f"SD of sample means:   {means.std():.4f} (theory {population_sigma/np.sqrt(n):.4f})")


## The “n >= 30” rule

The source uses $n\ge30$ for a non-Normal population. Treat this as a classroom heuristic.

- Mildly skewed population: 30 may be plenty.
- Strongly skewed or outlier-prone population: much more may be needed.
- Infinite-variance Pareto distribution: the classic finite-variance CLT does not apply.
- Dependent time-series or clustered observations: effective sample size may be much smaller.

Always inspect the process, not merely the number 30.

## Other CLT uses

- Sums: $S_n=\sum X_i\approx N(n\mu,n\sigma^2)$.
- Sample proportions: $\hat p\approx N(p,p(1-p)/n)$ when expected success and failure counts are large.
- Confidence intervals and hypothesis tests rely heavily on these approximations.

## Common mistakes

- Confusing SD of observations with SE of the mean.
- Forgetting the square root in $\sigma/\sqrt n$.
- Saying CLT works for any distribution without checking finite variance and dependence.
- Thinking repeated samples must physically be collected; sampling distributions can also be theoretical or simulated.


# 13 - Estimates

**Source alignment:** `13-Estimates.pdf`, page 1, and transcript lines 3777-3990. The source moves from descriptive to inferential statistics and distinguishes point estimates from interval estimates.

## From population to sample

- **Population:** the full group of interest.
- **Parameter:** a fixed but usually unknown population number, such as $\mu$, $p$, or $\sigma^2$.
- **Sample:** the observed subset.
- **Statistic:** a number calculated from the sample, such as $\bar x$.

Inferential statistics uses sample information to learn about population parameters.


## Estimator vs estimate

These words are related but different:

- **Estimator:** the rule or formula before seeing data, such as $\bar X$.
- **Estimate:** the numerical result after applying it, such as $\bar x=60$.

Examples:

| Parameter | Estimator | Point estimate after data |
|---|---|---|
| Population mean $\mu$ | Sample mean $\bar X$ | $\bar x$ |
| Population proportion $p$ | Sample proportion $\hat P$ | $\hat p=k/n$ |
| Population variance $\sigma^2$ | Sample variance $S^2$ | $s^2$ |


## 1. Point estimate

A point estimate is one numerical value used to estimate an unknown parameter.

Example from the source:

- unknown population mean: $\mu=65$ in the teaching picture;
- observed sample mean: $\bar x=60$;
- $60$ is the point estimate of $\mu$.

Point estimates are simple, but they hide uncertainty. A different random sample would usually give a different answer.

## 2. Interval estimate

An interval estimate gives a range of plausible parameter values.

$$\text{confidence interval}=\text{point estimate}\pm\text{margin of error}$$

The source illustrates a point estimate of 60 with an interval $[55,65]$.


## Confidence interval for a population mean

### Population SD known

$$\bar x\pm z_{\alpha/2}\frac{\sigma}{\sqrt n}$$

### Population SD unknown

For an approximately Normal population, or sufficiently large suitable sample:

$$\bar x\pm t_{\alpha/2,n-1}\frac{s}{\sqrt n}$$

Terms:

- $\bar x$: point estimate;
- $s/\sqrt n$: estimated standard error;
- critical value: chosen from the confidence level;
- margin of error: critical value $\times$ standard error.

Common two-sided Normal critical values:

| Confidence level | $z^*$ |
|---|---:|
| 90% | 1.645 |
| 95% | 1.960 |
| 99% | 2.576 |


## Worked confidence-interval example

A sample has:

- $n=64$;
- $\bar x=60$;
- $s=20$.

The estimated standard error is:

$$SE=\frac{20}{\sqrt{64}}=2.5$$

A 95% t interval uses $t^*\approx1.998$ with 63 degrees of freedom:

$$60\pm1.998(2.5)$$

$$\approx[55.00,65.00]$$

This reproduces the source's interval while showing where it comes from.


In [ ]:
n, xbar, s, confidence = 64, 60, 20, 0.95
alpha = 1 - confidence
critical = stats.t.ppf(1 - alpha/2, df=n-1)
se = s / np.sqrt(n)
margin = critical * se
ci = (xbar - margin, xbar + margin)

print(f"t critical value = {critical:.4f}")
print(f"standard error   = {se:.4f}")
print(f"margin of error  = {margin:.4f}")
print(f"95% CI            = [{ci[0]:.4f}, {ci[1]:.4f}]")


## Correct interpretation of a 95% confidence interval

Frequentist meaning:

> If we repeated the sampling method many times and built an interval each time, about 95% of those intervals would contain the true parameter.

After one interval is calculated, the parameter is fixed. Avoid saying “there is a 95% probability that $\mu$ is in this particular interval” unless using a Bayesian credible interval with an explicit prior and model.

## Confidence interval for a proportion

A simple large-sample interval is:

$$\hat p\pm z^*\sqrt{\frac{\hat p(1-\hat p)}{n}}$$

The basic Wald interval can behave badly for small samples or proportions near 0 or 1. Wilson or exact methods are often better.


## What makes an estimator good?

### Unbiasedness

$$E[\hat\theta]=\theta$$

It is correct on average over repeated samples.

### Consistency

$\hat\theta$ moves toward $\theta$ as sample size grows.

### Efficiency

Among comparable estimators, lower sampling variance is preferred.

### Robustness

The estimator is not badly damaged by small assumption violations or outliers.

### Mean squared error

$$\operatorname{MSE}(\hat\theta)=\operatorname{Var}(\hat\theta)+\operatorname{Bias}(\hat\theta)^2$$

A slightly biased estimator can have better overall MSE if it greatly reduces variance.


## Other estimation methods

### Maximum likelihood estimation (MLE)

Choose parameter values that make the observed data most likely under the model.

Examples:

- Bernoulli/Binomial: $\hat p=k/n$;
- Poisson: $\hat\lambda=\bar x$;
- Normal mean: $\hat\mu=\bar x$.

### Method of moments

Match sample moments, such as the sample mean, to theoretical moments.

### Bootstrap interval

Repeatedly resample the observed data with replacement, calculate the statistic, and use the empirical distribution to estimate uncertainty. Bootstrap validity still depends on the sample representing the population.

## Standard deviation vs standard error

- **SD:** variability among individual observations.
- **SE:** variability of an estimator across repeated samples.
- For the mean: $SE(\bar X)=\sigma/\sqrt n$ or estimated as $s/\sqrt n$.

## Common mistakes

- Confusing an estimator with its observed estimate.
- Reporting a point estimate without uncertainty.
- Interpreting confidence level as the fraction of data inside the interval.
- Assuming a narrow interval is accurate when sampling bias is present.
- Believing a larger sample fixes biased measurement or non-representative sampling.


# Master Cheatsheet

## One-table distribution summary

| Model | Support | Key conditions / story | Main formula | Mean | Variance |
|---|---|---|---|---:|---:|
| Bernoulli$(p)$ | $0,1$ | One binary trial | $p^x(1-p)^{1-x}$ | $p$ | $p(1-p)$ |
| Binomial$(n,p)$ | $0,\ldots,n$ | Fixed independent binary trials, same $p$ | $\binom nkp^k(1-p)^{n-k}$ | $np$ | $np(1-p)$ |
| Poisson$(\lambda)$ | $0,1,2,\ldots$ | Independent count at constant rate | $e^{-\lambda}\lambda^k/k!$ | $\lambda$ | $\lambda$ |
| Normal$(\mu,\sigma^2)$ | $\mathbb R$ | Symmetric additive variation | $\frac1{\sigma\sqrt{2\pi}}e^{-(x-\mu)^2/(2\sigma^2)}$ | $\mu$ | $\sigma^2$ |
| Std Normal | $\mathbb R$ | Normal after z-scoring | $z=(x-\mu)/\sigma$ | 0 | 1 |
| Continuous Uniform$(a,b)$ | $[a,b]$ | Constant density in bounds | $1/(b-a)$ | $(a+b)/2$ | $(b-a)^2/12$ |
| Discrete Uniform | $a,\ldots,b$ | $n=b-a+1$ equally likely integers | $1/n$ | $(a+b)/2$ | $(n^2-1)/12$ |
| Log-normal$(\mu,\sigma^2)$ | $x>0$ | $\ln X$ is Normal | see Ch. 09 | $e^{\mu+\sigma^2/2}$ | $(e^{\sigma^2}-1)e^{2\mu+\sigma^2}$ |
| Pareto$(x_m,\alpha)$ | $x\ge x_m$ | Power-law upper tail | $\alpha x_m^\alpha/x^{\alpha+1}$ | $\alpha x_m/(\alpha-1)$ if $\alpha>1$ | finite only if $\alpha>2$ |

## PMF, PDF, CDF in ten seconds

- PMF: discrete exact probability.
- PDF: continuous density; area gives probability.
- CDF: $F(x)=P(X\le x)$ for all distributions.
- Discrete: CDF = cumulative sum; PMF = jump size.
- Continuous: CDF = integral of PDF; PDF = derivative of CDF.

## Question wording translator

| Words | Probability form |
|---|---|
| exactly $k$ | $P(X=k)$ |
| at most $k$ | $P(X\le k)$ |
| fewer than $k$ | $P(X\le k-1)$ for integer counts |
| at least $k$ | $1-P(X\le k-1)$ |
| more than $k$ | $1-P(X\le k)$ |
| between $a$ and $b$ | CDF subtraction or sum/area |

## Formula triggers

- Fixed number of equal-probability yes/no trials -> Binomial.
- Count in time/space with constant rate -> Poisson.
- Convert a Normal value to SD units -> z-score.
- Sampling mean -> mean $\mu$, SE $\sigma/\sqrt n$.
- Confidence interval -> estimate $\pm$ critical value $\times$ SE.

## Red-flag checklist

- Probability must be between 0 and 1.
- A continuous exact point has probability 0.
- Z-scoring does not make data Normal.
- $n\ge30$ is a CLT heuristic, not a guarantee.
- 80/20 is not the definition of a power law or Pareto distribution.
- Large sample size reduces random error, not systematic bias.


# Important Revision Questions with Answers

## Q1. What is the main difference between a PMF and a PDF?

**Answer:** A PMF gives exact probabilities for discrete values: $P(X=x)$. A PDF gives density for a continuous variable; probability is area over an interval. For a continuous variable, $P(X=x)=0$.

## Q2. How do you obtain a CDF from a PMF or PDF?

**Answer:** For discrete $X$, sum PMF values up to $x$: $F(x)=\sum_{k\le x}p(k)$. For continuous $X$, integrate: $F(x)=\int_{-\infty}^x f(t)dt$.

## Q3. What four conditions make a Binomial model appropriate?

**Answer:** BINS: binary outcomes, independent trials, fixed number $n$, and same success probability $p$ on every trial.

## Q4. A Binomial variable has $n=10$ and $p=0.1$. What are its mean and variance?

**Answer:** Mean $np=1$. Variance $np(1-p)=10(0.1)(0.9)=0.9$.

## Q5. What does $\lambda=3$ mean in a Poisson model?

**Answer:** It means the expected or average count is 3 in the stated interval or exposure. It does not mean at least 3 events must occur.

## Q6. What does a z-score of $-1.5$ mean?

**Answer:** The value is 1.5 standard deviations below the mean.

## Q7. If $X\sim U(10,40)$, what is $P(15\le X\le30)$?

**Answer:** Wanted width divided by total width: $(30-15)/(40-10)=15/30=0.5$.

## Q8. If $X$ is Log-normal, what distribution does $\ln X$ follow?

**Answer:** A Normal distribution. The Log-normal parameters $\mu$ and $\sigma$ live on this log scale.

## Q9. What does the Central Limit Theorem make approximately Normal?

**Answer:** The sampling distribution of a properly standardised sum or sample mean, under suitable conditions. It does not make the raw population data Normal.

## Q10. What is the difference between a point estimate and an interval estimate?

**Answer:** A point estimate is one value, such as $\bar x=60$. An interval estimate gives a range plus a confidence level, such as a 95% CI $[55,65]$, to communicate sampling uncertainty.
